# Consolidated 1000 epigenomes

"Universal annotation of the human genome through integration of over a thousand epigenomic datasets"
https://link.springer.com/article/10.1186/s13059-021-02572-z

See https://egg2.wustl.edu/roadmap/data/byFileType/metadata/EID_metadata.tab

Analysis is available in `epi_1000.sh`

## Compute

In [ ]:
import glob
import importlib
import os
import pickle
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import Image, display
from tqdm.auto import tqdm

# Load configuration
config_path = os.path.abspath(os.path.expanduser("~/work/omni-chromhmm/config.yaml"))
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

project_root = os.path.dirname(config_path)
scripts_dir = os.path.join(project_root, "scripts", "analysis")
scripts_rules_dir = os.path.join(project_root, "scripts", "rules")

workdir = os.path.expanduser(os.path.expanduser("~/data/2026_epi_1000"))

# Import the analysis methods directly (no CLI / subprocess).
sys.path.insert(0, scripts_dir)
sys.path.insert(0, scripts_rules_dir)

import analyze
import analyze_peaks
import compare
import compare_methods
import compare_inter_dataset as compare_out
import emission_similarity
import match
import summary_plots
import utils
from tqdm.auto import tqdm
import importlib
import compare
import match
import pickle
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
from collections import defaultdict
from itertools import combinations

# Re-import the analysis modules so edits to scripts/analysis/*.py are picked up when this
# cell is re-run, without needing a kernel restart (plain `import` caches modules).
for _m in (analyze, analyze_peaks, compare, compare_methods,
           compare_out, emission_similarity,
           match, summary_plots, utils):
    importlib.reload(_m)

# Background states, excluded in the "noqh" variant of every metric below: the
# Quiescent/Heterochromatin bulk of the genome, whose agreement is both easy and
# uninformative. Each state model names them differently.
NOQH_DENOVO = {"Quies", "Het", "15_Quies", "9_Het"}   # de-novo 15-state models
NOQH_15 = {"15_Quies", "9_Het"}                       # ENCODE 15-state coreMarks
NOQH_18 = {"18_Quies", "13_Het"}                      # ENCODE 18-state core


def consistency_cache_path(slug, window):
    """Cache of analyze.compute_state_consistency() for one method and window."""
    return f"out/consistency/{slug}_w{window}.pkl"


In [ ]:
# Run everything relative to the pipeline working directory.
os.chdir(workdir)
print(f"Project root: {project_root}")
print(f"Scripts dir : {scripts_dir}")
print(f"Working dir : {workdir}")

# --- Parameters (mirrors the Snakefile) ----------------------------------
P = config["params"]
TOOLS = config["tools"]
DATASETS = config["datasets"]
MARKS = ["H3K36me3", "H3K9me3", "H3K4me1", "H3K27ac", "H3K27me3", "H3K4me3"]
CHROMHMM_BIN = P["chromhmm_bin"]
OMNI_BIN = P["omni_bin"]
HOMER_BIN = P["homer_bin"]
MACS2_BIN = P["macs2_bin"]
CALLER_BIN = {"omni": OMNI_BIN, "homer": HOMER_BIN, "macs2": MACS2_BIN}

DO_REPLICATES = P.get("replicates", False)

# Peak callers to include. Edit to match the segmentations you actually produced;
# missing files are skipped gracefully throughout the notebook.
CALLERS = ["homer", "macs2", "omni"]

COORDS_DIR = os.path.join(workdir, TOOLS["coords_dir"])
GENCODE_GTF = os.path.join(workdir, TOOLS["gencode_gtf"])
MARKUPS_DIR = os.path.join(project_root, "markups")

# De-novo methods compared across datasets
INTER_DS_METHODS = (
        ["chromhmm_default"]
        + [f"kmeans_{c}" for c in CALLERS]
        + [f"joint_kmeans_{c}" for c in CALLERS]
)

# --- Path helpers (mirror the Snakefile functions) -----------------------
def seg_bin(path):
    for caller, size in CALLER_BIN.items():
        if f"/{caller}/" in path:
            return size
    return CHROMHMM_BIN


print(f"Callers       : {CALLERS}")
print(f"Inter methods : {INTER_DS_METHODS}")


In [ ]:
EPI_1000_PATH = os.path.expanduser('~/data/2026_epi_1000')
os.makedirs("out", exist_ok=True)

ref_path = os.path.expanduser("~/data/2026_omni_chromhmm/monocytes/ENCFF227EMB_chromhmm.bed")
ref_segs = match.load_bed(ref_path)
ref_colors = match.state_colors(ref_segs)

exxx_folders = sorted(
    [d for d in os.listdir(EPI_1000_PATH) if d.startswith("E") and os.path.isdir(os.path.join(EPI_1000_PATH, d))])

from functools import partial

methods = {
    "ChromHMM": "{f}_chromhmm/{f}_15_dense_matched.bed",
    "HOMER": "homer/{f}_homer_kmeans_states_matched.bed",
    "MACS2": "macs2/{f}_macs2_kmeans_states_matched.bed",
    "Omnipeak": "omni/{f}_omni_kmeans_states_matched.bed",
    "Joint HOMER": "../joint_kmeans/homer/{f}_kmeans_joint_states_matched.bed",
    "Joint MACS2": "../joint_kmeans/macs2/{f}_kmeans_joint_states_matched.bed",
    "Joint Omnipeak": "../joint_kmeans/omni/{f}_kmeans_joint_states_matched.bed"
}

# Prepare tasks
tasks = []
for folder in exxx_folders:
    f_path = os.path.join(EPI_1000_PATH, folder)
    for method_name, pt in methods.items():
        p = os.path.join(f_path, pt.format(f=folder))
        if os.path.exists(p):
            tasks.append((folder, method_name, p, seg_bin(p)))

# Consolidate results using ProcessPoolExecutor with script functions
valid_tasks = [t for t in tasks if os.path.exists(t[2])]
paths = [t[2] for t in valid_tasks]
bins = [t[3] for t in valid_tasks]

def check_valid_worker(path):
    try:
        segs = match.load_bed(path)
        return all(s[3] and s[3] != '.' for s in segs)
    except Exception:
        return False

print(f"Processing {len(valid_tasks)} existing segmentations...")
with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
    print("Checking validity of segmentations...")
    valid_mask = list(tqdm(executor.map(check_valid_worker, paths), total=len(paths)))

print('Identify and filter out segmentations with non-matched states (missing names)')
matched_indices = [i for i, is_valid in enumerate(valid_mask) if is_valid]
if len(matched_indices) < len(valid_mask):
    print(f"Filtering out {len(valid_mask) - len(matched_indices)} segmentations with missing state names.")
    for i in set(range(len(valid_mask))) - set(matched_indices):
        folder, method_name, _, _ = valid_tasks[i]
        print(f"  - {folder} {method_name}")
    
    valid_tasks = [valid_tasks[i] for i in matched_indices]
    paths = [paths[i] for i in matched_indices]
    bins = [bins[i] for i in matched_indices]

# Store paths in all_segs instead of full content to save memory
all_segs = paths

def tm_worker(i):
    segs = match.load_bed(all_segs[i])
    return analyze.build_transition_matrix(segs, bins[i])

def tm_noqh_worker(i):
    exclude_noqh = {"Quies", "Het", "15_Quies", "9_Het"}
    segs = match.load_bed(all_segs[i])
    return analyze.build_transition_matrix(segs, bins[i],
                                           exclude_states=exclude_noqh)

def stats_worker(i):
    segs = match.load_bed(all_segs[i])
    return compare.compute_segment_stats(segs)

In [ ]:
def compute_transition_matrices():
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        full = list(tqdm(executor.map(tm_worker, range(len(all_segs))), total=len(all_segs)))
        noqh = list(tqdm(executor.map(tm_noqh_worker, range(len(all_segs))), total=len(all_segs)))
    return full, noqh


# A cache from a different set of segmentations is not reusable at all.
all_tm, all_tm_noqh = utils.cached_pickle(
    "out/all_tm.pkl", compute_transition_matrices, label="transition matrices",
    valid=lambda cached: len(cached[0]) == len(valid_tasks))


In [ ]:
def compute_segmentation_stats():
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        return list(tqdm(executor.map(stats_worker, range(len(all_segs))), total=len(all_segs)))


all_stats = utils.cached_pickle(
    "out/all_stats.pkl", compute_segmentation_stats, label="segmentation statistics",
    valid=lambda cached: len(cached) == len(valid_tasks))


In [ ]:
def compute_results():
    results = []
    for i in range(len(valid_tasks)):
        folder, method_name, _, _ = valid_tasks[i]
        states, counts, s_bp = all_tm[i]
        entropy, _, _, _ = analyze.transition_entropy(states, counts, s_bp)
        states_n, counts_n, s_bp_n = all_tm_noqh[i]
        entropy_noqh = analyze.transition_entropy(states_n, counts_n, s_bp_n)[0] if states_n else 0
        results.append({
            "Dataset": folder,
            "Method": method_name,
            "N_States": all_stats[i].get("n_states", 0),
            "N_Segments": all_stats[i].get("n_segments", 0),
            "Entropy": entropy,
            "Entropy_NOQH": entropy_noqh
        })
    return pd.DataFrame(results)

df_results = utils.cached_csv(
    "out/df_results.csv", compute_results, label="results",
    valid=lambda df: len(df) == len(valid_tasks))

In [ ]:
def state_comp_worker(idx):
    folder, method_name, path, _ = valid_tasks[idx]
    try:
        segs = match.load_bed(path)
        lengths_by_state = defaultdict(list)
        for row in segs:
            name = row[3]
            if not name or name == '.': continue
            lengths_by_state[name].append(row[2] - row[1])
        
        tot_bp = sum(sum(ls) for ls in lengths_by_state.values())
        
        res = []
        for st, ls in lengths_by_state.items():
            res.append({
                "Dataset": folder,
                "Method": method_name,
                "State": st,
                "Fraction": sum(ls) / tot_bp if tot_bp > 0 else 0,
                "MeanLength": np.mean(ls),
                "MedianLength": np.median(ls)
            })
        return res
    except Exception as e:
        # print(f"Error processing {path}: {e}")
        return []

def compute_state_comp():
    print('Computing state composition and lengths...')
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        results_list = list(tqdm(executor.map(state_comp_worker, range(len(valid_tasks))), total=len(valid_tasks)))
    
    compositions = [item for sublist in results_list for item in sublist]
    return pd.DataFrame(compositions)

df_comp = utils.cached_csv(
    "out/df_comp.csv", compute_state_comp, label="state composition",
    valid=lambda df: len(df.groupby(["Dataset", "Method"])) == len(valid_tasks) and "MeanLength" in df.columns)

In [ ]:
# 7. Pairwise consistency (Jaccard and Kappa)
method_order = ["ChromHMM", "HOMER", "MACS2", "Omnipeak", "Joint HOMER", "Joint MACS2", "Joint Omnipeak"]

def state_lengths_worker(idx):
    return match.state_lengths(match.load_bed(all_segs[idx]))

def pair_overlap_worker(t):
    m_name, i, j = t
    return match.pair_overlap(None, match.load_bed(all_segs[j]), ref_index=current_indices[i])

all_lengths = {}
df_pw_list = []

for m_name in method_order:
    cache_path = f"out/pw_cache_{m_name.lower()}.pkl"
    # ! rm {cache_path}
    if os.path.exists(cache_path):
        print(f"Loading cached pairwise consistency for {m_name} from {cache_path}...")
        with open(cache_path, "rb") as f:
            method_cache = pickle.load(f)
            df_pw_list.append(method_cache['df'])
            # Populate lengths just in case, although if we have df we don't strictly need them here
            if 'lengths' in method_cache:
                all_lengths.update(method_cache['lengths'])
            continue

    # Otherwise compute
    idxs = [idx for idx, task in enumerate(valid_tasks) if task[1] == m_name]
    missing_idxs = [i for i in idxs if i not in all_lengths]
    if missing_idxs:
        print(f"Computing lengths for {m_name} ({len(missing_idxs)} samples)...")
        with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
            new_lengths = list(tqdm(executor.map(state_lengths_worker, missing_idxs, chunksize=10), total=len(missing_idxs)))
        for i, lengths in zip(missing_idxs, new_lengths):
            all_lengths[i] = lengths

    pairwise_tasks = [(m_name, i, j) for i, j in combinations(idxs, 2)]
    if len(pairwise_tasks) > 1000:
        random.seed(42)
        pairwise_tasks = random.sample(pairwise_tasks, 1000)
        pairwise_tasks.sort()

    if not pairwise_tasks:
        print(f"No pairwise tasks for {m_name}, skipping.")
        continue

    print(f"Computing pairwise overlaps for {m_name} ({len(pairwise_tasks)} pairs)...")
    # Pre-build indices for the current method's segmentations to avoid redundant work and memory bloat
    unique_idxs = {t[1] for t in pairwise_tasks}
    current_indices = {i: match.build_index(match.load_bed(all_segs[i])) for i in tqdm(unique_idxs, desc="Indexing", leave=False)}

    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        overlaps = list(tqdm(executor.map(pair_overlap_worker,
                                          pairwise_tasks,
                                          chunksize=100),
                             total=len(pairwise_tasks)))
    del current_indices # Free memory

    pw_data = []
    for (m_name, i, j), overlap in zip(pairwise_tasks, overlaps):
        metrics = match.agreement_by_mode(overlap, all_lengths[i], all_lengths[j],
                                          background=NOQH_DENOVO)
        for mode, m in metrics.items():
            pw_data.append({"Method": m_name, "Mode": mode.upper(),
                            "Jaccard": m["Jaccard"], "Kappa": m["Kappa"]})

    df_method = pd.DataFrame(pw_data)
    print(f"Saving updated cache to {cache_path}...")
    method_lengths = {i: all_lengths[i] for i in idxs}
    with open(cache_path, "wb") as f:
        pickle.dump({'overlaps': overlaps, 'df': df_method, 'lengths': method_lengths}, f)
    df_pw_list.append(df_method)

# Final aggregation
if df_pw_list:
    df_pw = pd.concat(df_pw_list, ignore_index=True)
    # Ensure correct order for plotting
    df_pw['Method'] = pd.Categorical(df_pw['Method'], categories=method_order, ordered=True)
    df_pw = df_pw.sort_values(['Method', 'Mode']).reset_index(drop=True)
else:
    df_pw = pd.DataFrame(columns=["Method", "Mode", "Jaccard", "Kappa"])


In [ ]:
cache_peaks = "out/df_peaks.csv"

def process_peaks_worker(folder):
    f_path = os.path.join(EPI_1000_PATH, folder)
    outdir = os.path.join(f_path, "peaks")
    analyze_peaks.run_analyze_peaks(
        f_path, folder, list(MARKS), outdir,
        omni_bin=P["omni_bin"], chromhmm_bin=CHROMHMM_BIN
    )
    tsv = os.path.join(outdir, "peak_stats.tsv")
    if os.path.exists(tsv):
        df = pd.read_csv(tsv, sep="	")
        df["dataset"] = folder
        return df
    return None

def compute_peak_stats():
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context('fork')) as executor:
        dfs = list(tqdm(executor.map(process_peaks_worker, exxx_folders),
                        total=len(exxx_folders), desc="Peak stats"))
    return pd.concat([d for d in dfs if d is not None], ignore_index=True)


df_peaks = utils.cached_csv(cache_peaks, compute_peak_stats, label="peak statistics")


## Plotting

In [ ]:
method_palette = {
    "ChromHMM": summary_plots.BIN_COLORS["default"],
    "HOMER": summary_plots.BIN_COLORS["homer"],
    "MACS2": summary_plots.BIN_COLORS["macs2"],
    "Omnipeak": summary_plots.BIN_COLORS["omnipeak"],
    "Joint HOMER": summary_plots.BIN_COLORS["homer"],
    "Joint MACS2": summary_plots.BIN_COLORS["macs2"],
    "Joint Omnipeak": summary_plots.BIN_COLORS["omnipeak"]
}

In [ ]:
# Peak counts and mean peak length (exclude 5-95% outliers)
summary_plots.log_peak_outliers(df_peaks)
summary_plots._plot_peak_count(df_peaks, EPI_1000_PATH, "out/n_peaks.png")
summary_plots._plot_peak_length(df_peaks, EPI_1000_PATH, "out/peak_length.png")

In [ ]:
# 1. Number of states
fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(data=df_results, x="Method", y="N_States", hue="Method", palette=method_palette, order=list(methods.keys()),
            capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_results, x="Method", y="N_States",
                   order=list(methods.keys()), dodge=False, size=2)
ax.set_title("Number of unique matched states per method", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of states", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    vals = df_results[df_results["Method"] == method]["N_States"]
    m, s = vals.mean(), vals.sem()
    if pd.isna(m): continue
    top = max(m + (s if not pd.isna(s) else 0), vals.max())
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/n_states.png", bbox_inches="tight")
plt.close(fig)

In [ ]:
# 2. Number of segments
fig, ax = plt.subplots(figsize=(6, 4.2))
# Divide by 1000 for consistency with analysis.ipynb
df_results_copy = df_results.copy()
df_results_copy["N_Segments_K"] = df_results_copy["N_Segments"] / 1000.0
sns.barplot(data=df_results_copy, x="Method", y="N_Segments_K", hue="Method", palette=method_palette,
            order=list(methods.keys()), capsize=0.05,
            errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_results_copy, x="Method", y="N_Segments_K",
                   order=list(methods.keys()), dodge=False, size=2)
ax.set_title("Number of segments per method", fontsize=11, fontweight="bold")
ax.set_ylabel("Segments (×10³)", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    vals = df_results_copy[df_results_copy["Method"] == method]["N_Segments_K"]
    m, s = vals.mean(), vals.sem()
    if pd.isna(m): continue
    top = max(m + (s if not pd.isna(s) else 0), vals.max())
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/n_segments.png", bbox_inches="tight")
plt.close(fig)

In [ ]:
# 3. States composition (average per type)
# summary_plots.sort_states puts numbered names ('1_TssA', '10_TssB') in state order.
states_order = summary_plots.sort_states(df_comp['State'].unique())

# Ensure all states are present for each (Method, Dataset) pair, filling missing Fractions with 0
df_comp_filled = df_comp.pivot_table(index=['Method', 'Dataset'], columns='State', values='Fraction', fill_value=0).stack().reset_index(name='Fraction')

BREAK_LOW = 0.20
BREAK_HIGH = 0.40
figw = max(12, len(states_order) * len(method_palette) * 0.22)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(figw, 6),
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.06},
)

for ax in (ax_top, ax_bot):
    sns.barplot(
        data=df_comp_filled, x="State", y="Fraction", hue="Method",
        order=states_order, hue_order=list(methods.keys()), palette=method_palette,
        ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0},
        legend=(ax is ax_top), edgecolor="lightgrey", linewidth=1
    )
    utils.strip_points(ax, data=df_comp_filled, x="State", y="Fraction", hue="Method",
                       order=states_order, hue_order=list(methods.keys()),
                       size=1.5, alpha=0.4, jitter=0.2)

ax_top.set_ylim(BREAK_HIGH, 1.02)
ax_bot.set_ylim(0, BREAK_LOW)

# Hide the inner spines to create the visual break
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", bottom=False)

# Draw diagonal break marks
d = 0.012
kwargs = dict(transform=fig.transFigure, color="k", clip_on=False, linewidth=0.8)
for ax, sign in [(ax_top, -1), (ax_bot, 1)]:
    x0, x1 = ax.get_position().x0, ax.get_position().x1
    y = ax.get_position().y0 if sign == 1 else ax.get_position().y1
    for x in (x0, x1):
        fig.add_artist(plt.Line2D([x - d, x + d], [y + sign * d * 1.5, y - sign * d * 1.5], **kwargs))

for ax in (ax_top, ax_bot):
    ax.grid(axis="y", alpha=0.3, linewidth=0.5)
    ax.tick_params(axis="y", labelsize=8)

ax_bot.tick_params(axis="x", labelsize=8, rotation=45)
ax_bot.set_xlabel("Chromatin state", fontsize=9)
ax_top.set_xlabel("")
ax_top.set_title("Average state composition per method", fontsize=10, fontweight="bold")

ax_top.legend(title="Method", fontsize=8, title_fontsize=9,
              bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
if ax_bot.get_legend():
    ax_bot.get_legend().remove()

ax_top.set_ylabel("")
ax_bot.set_ylabel("Fraction of genome", fontsize=9)

plt.savefig("out/avg_composition.png", bbox_inches='tight')
plt.close(fig)


In [ ]:
# 3b. Average state mean length per method
fig, ax = plt.subplots(figsize=(figw, 6))
sns.barplot(
    data=df_comp, x="State", y="MeanLength", hue="Method",
    order=states_order, hue_order=list(methods.keys()), palette=method_palette,
    ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, edgecolor="lightgrey", linewidth=1
)
utils.strip_points(ax, data=df_comp, x="State", y="MeanLength", hue="Method",
                   order=states_order, hue_order=list(methods.keys()),
                   size=1.5, alpha=0.4, jitter=0.2)
ax.set_yscale("log")
ax.set_title("Average state mean length per method", fontsize=10, fontweight="bold")
ax.set_ylabel("Mean length (bp, log scale)", fontsize=9)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.tick_params(axis="x", labelsize=8, rotation=45)
ax.legend(title="Method", fontsize=8, title_fontsize=9,
          bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
fig.tight_layout()
fig.savefig("out/avg_mean_length.png", bbox_inches="tight")
plt.show()

# 3c. Average state median length per method
fig, ax = plt.subplots(figsize=(figw, 6))
sns.barplot(
    data=df_comp, x="State", y="MedianLength", hue="Method",
    order=states_order, hue_order=list(methods.keys()), palette=method_palette,
    ax=ax, capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, edgecolor="lightgrey", linewidth=1
)
utils.strip_points(ax, data=df_comp, x="State", y="MedianLength", hue="Method",
                   order=states_order, hue_order=list(methods.keys()),
                   size=1.5, alpha=0.4, jitter=0.2)
ax.set_yscale("log")
ax.set_title("Average state median length per method", fontsize=10, fontweight="bold")
ax.set_ylabel("Median length (bp, log scale)", fontsize=9)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
ax.tick_params(axis="x", labelsize=8, rotation=45)
ax.legend(title="Method", fontsize=8, title_fontsize=9,
          bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)
fig.tight_layout()
fig.savefig("out/avg_median_length.png", bbox_inches="tight")
plt.show()

In [ ]:
# 4. States composition for each method individually
# Collect state colors from all loaded segmentations for better matching
state_colors_hex = {}
# Only sample a few segmentations per method to get colors efficiently
sampled_idxs = []
for m_name in methods.keys():
    m_idxs = [idx for idx, task in enumerate(valid_tasks) if task[1] == m_name]
    if m_idxs:
        sampled_idxs.extend(m_idxs[:3]) # Take up to 3 samples per method

for i in sampled_idxs:
    segs = match.load_bed(all_segs[i])
    for row in segs:
        name = row[3]
        color = row[4] if len(row) > 4 else "0,0,0"
        if name not in state_colors_hex and color != "0,0,0":
            state_colors_hex[name] = analyze.rgb_str_to_hex(color)

# Fallback to ref_colors and summary_plots.STATE_COLORS
for s in states_order:
    if s not in state_colors_hex or state_colors_hex[s] == "#000000":
        # Try exact match in ref_colors
        if s in ref_colors:
            state_colors_hex[s] = analyze.rgb_str_to_hex(ref_colors[s])
        else:
            # Try prefix match in summary_plots.STATE_COLORS
            name_part = s.split('_')[1] if '_' in s else s
            found_color = None
            for canonical, rgb in summary_plots.STATE_COLORS.items():
                if name_part.startswith(canonical):
                    found_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                    break
            if found_color:
                state_colors_hex[s] = found_color
            else:
                state_colors_hex[s] = analyze.rgb_str_to_hex(ref_colors.get(s, "128,128,128"))

for method in methods.keys():
    method_df = df_comp[df_comp["Method"] == method]
    if method_df.empty: continue
    pivot_df = method_df.pivot(index="Dataset", columns="State", values="Fraction").fillna(0)
    avail_states = [s for s in states_order if s in pivot_df.columns]
    pivot_df = pivot_df[avail_states]
    colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

    ax = pivot_df.plot(kind='bar', stacked=True, figsize=(20, 8), color=colors, width=0.8, linewidth=0)
    pivot_df.sum(axis=1).plot(kind='bar', ax=ax, width=0.8, facecolor='none', edgecolor="lightgrey", linewidth=1, legend=False)
    ax.set_title(f"State composition per dataset - {method}", fontsize=11, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='small', title="State")
    ax.set_xlabel("Dataset (EID)", fontsize=9)
    ax.set_ylabel("Fraction of Genome", fontsize=9)
    if len(pivot_df) > 50:
        ax.tick_params(axis='x', labelsize=6)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(f"out/composition_{method.lower()}.png", bbox_inches="tight")
    plt.close(fig)

In [ ]:
# 5. Transition matrix Entropy
fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(data=df_results, x="Method", y="Entropy", hue="Method", palette=method_palette, order=list(methods.keys()),
            capsize=0.05,
            errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_results, x="Method", y="Entropy",
                   order=list(methods.keys()), dodge=False, size=2)
ax.set_title("Transition matrix entropy (full)", fontsize=11, fontweight="bold")
ax.set_ylabel("Entropy (bits)", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    vals = df_results[df_results["Method"] == method]["Entropy"]
    m, s = vals.mean(), vals.sem()
    if pd.isna(m): continue
    top = max(m + (s if not pd.isna(s) else 0), vals.max())
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/entropy.png", bbox_inches="tight")
plt.close(fig)

# 5b. Transition matrix Entropy (NOQH)
fig, ax = plt.subplots(figsize=(6, 4.2))
sns.barplot(data=df_results, x="Method", y="Entropy_NOQH", hue="Method", palette=method_palette,
            order=list(methods.keys()), capsize=0.05,
            errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_results, x="Method", y="Entropy_NOQH",
                   order=list(methods.keys()), dodge=False, size=2)
ax.set_title("Transition matrix entropy (NOQH, Excl. Quies/Het)", fontsize=11, fontweight="bold")
ax.set_ylabel("Entropy (bits)", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
# Add labels over error bars
method_order = list(methods.keys())
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
for i, method in enumerate(method_order):
    vals = df_results[df_results["Method"] == method]["Entropy_NOQH"]
    m, s = vals.mean(), vals.sem()
    if pd.isna(m): continue
    top = max(m + (s if not pd.isna(s) else 0), vals.max())
    ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
fig.tight_layout()
fig.savefig("out/entropy_noqh.png", bbox_inches="tight")
plt.close(fig)

In [ ]:
# 6. Summary average state composition per method (stacked)
pivot_avg = df_comp.pivot_table(index=['Method', 'Dataset'], columns='State', values='Fraction', fill_value=0).groupby('Method').mean()
# Reorder by methods keys to keep consistent order
pivot_avg = pivot_avg.reindex(list(methods.keys()))
avail_states = [s for s in states_order if s in pivot_avg.columns]
pivot_avg = pivot_avg[avail_states]
colors = [state_colors_hex.get(s, "#888888") for s in avail_states]

plt.figure(figsize=(8, 5))
ax = pivot_avg.plot(kind='bar', stacked=True, color=colors, width=0.6, ax=plt.gca(), linewidth=0)
pivot_avg.sum(axis=1).plot(kind='bar', ax=ax, width=0.6, facecolor='none', edgecolor="lightgrey", linewidth=1, legend=False)
ax.set_title("Average state composition per method", fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='small', title="State")
ax.set_xlabel("Method", fontsize=9)
ax.set_ylabel("Average Fraction of Genome", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)
plt.tight_layout()
plt.savefig("out/avg_composition_stacked.png", bbox_inches="tight")
plt.close()

In [ ]:
# Plotting
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        fig, ax = plt.subplots(figsize=(6, 4.5))
        df_mode = df_pw[df_pw["Mode"] == mode]
        sns.barplot(data=df_mode, x="Method", y=metric, hue="Method", palette=method_palette,
                    order=list(methods.keys()),
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax, edgecolor="lightgrey", linewidth=1)
        utils.strip_points(ax, data=df_mode, x="Method", y=metric,
                           order=list(methods.keys()), dodge=False,
                           size=1, alpha=0.25)
        ax.set_title(f"Pairwise {metric} consistency ({mode})", fontsize=11, fontweight="bold")
        ax.set_ylabel(f"{metric} index", fontsize=9)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
        ax.grid(axis="y", alpha=0.3)

        # Add labels over error bars
        method_order = list(methods.keys())
        yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
        for i, method in enumerate(method_order):
            subset = df_mode[df_mode["Method"] == method][metric]
            if subset.empty: continue
            m, s = subset.mean(), subset.sem()
            top = utils.bar_label_y(ax, m + (s if not pd.isna(s) else 0), subset.max())
            ax.text(i, top, f"{m:.2f}", ha="center", va="bottom", fontsize=6)

        fig.tight_layout()
        fig.savefig(f"out/pairwise_consistency_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
        plt.close(fig)

In [ ]:
# 8. Pairwise state-by-state consistency heatmaps
print("Computing state-by-state consistency heatmaps...")
for mode in ["full", "noqh"]:
    for metric in ["Jaccard", "Kappa"]:
        for m_name in method_order:
            cache_path = f"out/pw_cache_{m_name.lower()}.pkl"
            if not os.path.exists(cache_path):
                continue
            with open(cache_path, "rb") as f:
                cache = pickle.load(f)

            overlaps = cache.get('overlaps', [])
            lengths = cache.get('lengths', {})
            if not overlaps:
                continue

            # Re-generate pairwise_tasks to match overlaps with lengths
            idxs = [idx for idx, task in enumerate(valid_tasks) if task[1] == m_name]
            pairwise_tasks = [(m_name, i, j) for i, j in combinations(idxs, 2)]
            if len(pairwise_tasks) > 1000:
                random.seed(42)
                pairwise_tasks = random.sample(pairwise_tasks, 1000)
                pairwise_tasks.sort()

            excl = NOQH_DENOVO if mode == "noqh" else set()
            avail_states = [s for s in states_order if s in lengths.get(idxs[0], {}) and s not in excl]

            # Cells are averaged only over the pairs where they are *defined*.
            # A state absent from both segmentations of a pair carries no
            # information; averaging it in as 0.0 (the old behaviour) dragged the
            # mean down in proportion to how often the state went missing — up to
            # -0.23 on the Joint HOMER diagonal, where 84 distinct state sets
            # occur across the 1000 pairs. A state present on one side only keeps
            # its legitimate 0 (called by one, never by the other).
            sum_mat = np.zeros((len(avail_states), len(avail_states)))
            n_mat = np.zeros((len(avail_states), len(avail_states)))
            count = 0

            for (mn, i, j), overlap in zip(pairwise_tasks, overlaps):
                # Restricted marginals and total area for the current set of states
                total_area = sum(overlap.get((s1, s2), 0) for s1 in avail_states for s2 in avail_states)
                if total_area == 0: continue
                a1_s = {s: sum(overlap.get((s, s2), 0) for s2 in avail_states) for s in avail_states}
                a2_s = {s: sum(overlap.get((s1, s), 0) for s1 in avail_states) for s in avail_states}

                mat = np.full((len(avail_states), len(avail_states)), np.nan)
                if metric == "Jaccard":
                    for row_idx, s1 in enumerate(avail_states):
                        for col_idx, s2 in enumerate(avail_states):
                            ov = overlap.get((s1, s2), 0)
                            union = a1_s[s1] + a2_s[s2] - ov
                            if union > 0:
                                mat[row_idx, col_idx] = ov / union
                else: # Kappa
                    for row_idx, s1 in enumerate(avail_states):
                        p1 = a1_s[s1] / total_area
                        for col_idx, s2 in enumerate(avail_states):
                            p2 = a2_s[s2] / total_area
                            p12 = overlap.get((s1, s2), 0) / total_area
                            denom = p1 + p2 - 2 * p1 * p2
                            if denom > 0:
                                mat[row_idx, col_idx] = 2 * (p12 - p1 * p2) / denom

                defined = ~np.isnan(mat)
                sum_mat[defined] += mat[defined]
                n_mat += defined
                count += 1

            if count > 0:
                avg_mat = np.where(n_mat > 0, sum_mat / np.maximum(n_mat, 1), np.nan)
                n_diag = np.diag(n_mat).astype(int)
                if n_diag.min() < count:
                    thin = {s: int(n) for s, n in zip(avail_states, n_diag) if n < count}
                    print(f"  {m_name} {mode.upper()} {metric}: diagonal averaged over "
                          f"fewer than {count} pairs where the state was absent from both "
                          f"segmentations: {thin}")
                plt.figure(figsize=(10, 8))
                sns.heatmap(avg_mat, annot=True, fmt=".2f",
                            cmap="Reds" if metric == "Jaccard" else "RdBu_r",
                            center=0 if metric == "Kappa" else None,
                            xticklabels=avail_states, yticklabels=avail_states,
                            annot_kws={"size": 6})
                subtitle = ("one-vs-rest 2x2 kappa per state, own 0-1 scale"
                            if metric == "Kappa" else
                            "pairwise Jaccard per state pair")
                plt.title(f"{m_name} Average {metric} Matrix ({mode.upper()})\n{subtitle}", fontsize=10)
                plt.xlabel("State (Segmentation 2)")
                plt.ylabel("State (Segmentation 1)")
                plt.tight_layout()
                plt.savefig(f"out/pw_heatmap_{m_name.lower().replace(' ', '_')}_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
                plt.close()


## Results

### 1. De-novo methods comparison

In [ ]:
for img in ["n_peaks.png", "peak_length.png", "n_states.png", "n_segments.png", "avg_composition.png",
            "avg_mean_length.png", "avg_median_length.png", "avg_composition_stacked.png", 
            "entropy.png", "entropy_noqh.png"]:
    path = f"out/{img}"
    if os.path.exists(path):
        display(Image(filename=path))

### 2. 18-state core K27ac reference

In [ ]:
for img in ["epi_18_core_avg_composition.png", "epi_18_core_avg_mean_length.png", "epi_18_core_avg_median_length.png"]:
    path = f"out/{img}"
    if os.path.exists(path):
        display(Image(filename=path))
![image.png](attachment:dc20543d-0738-4ad0-9ec7-d73d15e90c76.png)#%% md
### 3. 15-state reference models (Individual vs Joint)

In [ ]:
for img in ["epi_15_avg_composition_comparison.png", "epi_15_avg_composition_stacked.png", 
            "epi_15_avg_mean_length_comparison.png", "epi_15_avg_median_length_comparison.png"]:
    path = f"out/{img}"
    if os.path.exists(path):
        display(Image(filename=path))

### 4. Pairwise consistency & Other

In [ ]:
for img in ["pairwise_consistency_full_jaccard.png", "pairwise_consistency_full_kappa.png", 
            "pairwise_consistency_noqh_jaccard.png", "pairwise_consistency_noqh_kappa.png"]:
    path = f"out/{img}"
    if os.path.exists(path):
        display(Image(filename=path))
for method in methods.keys():
    path = f"out/composition_{method.lower()}.png"
    if os.path.exists(path):
        display(Image(filename=path))

### 5. Pairwise state-by-state consistency heatmaps

In [ ]:
for mode in ["full", "noqh"]:
    for metric in ["jaccard", "kappa"]:
        print(f"### {metric.capitalize()} ({mode.upper()})")
        for m_name in method_order:
            path = f"out/pw_heatmap_{m_name.lower().replace(' ', '_')}_{mode}_{metric}.png"
            if os.path.exists(path):
                print(f"#### {m_name}")
                display(Image(filename=path))

# Reference 18-state core K27ac segmentations

### 1. Identify files and compute/load stats

In [ ]:
# 1. Identify files
epi_18_core_files = sorted(glob.glob(f"{EPI_1000_PATH}/E*_18_core_K27ac_dense.bed.gz"))
epi_18_core_ids = [os.path.basename(f).split("_")[0] for f in epi_18_core_files]

import json

def get_file_stats_18(f, segs):
    """(id, stats) of one 18-state segmentation - see analyze.segmentation_stats."""
    return os.path.basename(f).split("_")[0], analyze.segmentation_stats(
        segs, 200, background=NOQH_18)


# 2. Computation
results_18 = []
comp_18 = []
entropy_18 = []
state_colors_18 = {}
loaded_segs_18 = {}

# Check if all result files exist to skip processing
result_files = ["out/df_segments_18.csv", "out/df_comp_18.csv", "out/df_entropy_18.csv", "out/state_colors_18.json"]
all_on_disk = all(os.path.exists(f) for f in result_files)

if all_on_disk:
    print("Loading 18-state data from disk...")
    df_segments_18 = pd.read_csv("out/df_segments_18.csv")
    df_comp_18 = pd.read_csv("out/df_comp_18.csv")
    df_entropy_18 = pd.read_csv("out/df_entropy_18.csv")
    with open("out/state_colors_18.json", "r") as f:
        state_colors_18 = json.load(f)

print(f"Processing {len(epi_18_core_files)} 18-state files...")
for f, eid in tqdm(zip(epi_18_core_files, epi_18_core_ids), total=len(epi_18_core_files)):
    segs = match.load_bed(f)
    loaded_segs_18[eid] = segs

    if not all_on_disk:
        eid, stats = get_file_stats_18(f, segs)
        results_18.append({"Dataset": eid, "N_Segments": stats["n_segments"]})
        for c in stats["composition"]:
            comp_18.append({"Dataset": eid, "State": c["State"], "Fraction": c["Fraction"],
                            "MeanLength": c["MeanLength"], "MedianLength": c["MedianLength"]})
        for m, val in stats["entropy"].items():
            entropy_18.append({"Dataset": eid, "Mode": m.upper(), "Entropy": val})
        state_colors_18.update(stats["colors"])

if not all_on_disk:
    df_segments_18 = pd.DataFrame(results_18)
    df_comp_18 = pd.DataFrame(comp_18)
    df_entropy_18 = pd.DataFrame(entropy_18)

    # Save results
    df_segments_18.to_csv("out/df_segments_18.csv", index=False)
    df_comp_18.to_csv("out/df_comp_18.csv", index=False)
    df_entropy_18.to_csv("out/df_entropy_18.csv", index=False)
    with open("out/state_colors_18.json", "w") as f:
        json.dump(state_colors_18, f)

# 3. Pairwise metrics
def compute_pw_18():
    print("Computing pairwise overlaps for (1000 pairs)...")
    loaded_lengths = {eid: match.state_lengths(segs) for eid, segs in loaded_segs_18.items()}
    pairs = [tuple(sorted((id1, id2))) for i, id1 in enumerate(epi_18_core_ids) for id2 in epi_18_core_ids[i+1:]]
    if len(pairs) > 1000:
        random.seed(42)
        pairs = random.sample(pairs, 1000)
        pairs.sort()

    pw_data_18 = []
    overlaps_18 = []
    for id1, id2 in tqdm(pairs):
        overlap = match.pair_overlap(loaded_segs_18[id1], loaded_segs_18[id2])
        overlaps_18.append(overlap)
        metrics = match.agreement_by_mode(overlap, loaded_lengths[id1], loaded_lengths[id2],
                                          background=NOQH_18)
        for mode, m in metrics.items():
            pw_data_18.append({"Dataset1": id1, "Dataset2": id2, "Mode": mode.upper(),
                               "Jaccard": m["Jaccard"], "Kappa": m["Kappa"]})
    df_pw_18 = pd.DataFrame(pw_data_18)
    df_pw_18.to_csv("out/df_pw_18.csv", index=False)
    return {'df': df_pw_18, 'overlaps': overlaps_18}

cache_pw_18_path = "out/pw_18_core_cache.pkl"
cache_18 = utils.cached_pickle(cache_pw_18_path, compute_pw_18, label="18-state pairwise metrics")
df_pw_18 = cache_18['df']
overlaps_18 = cache_18['overlaps']


### 2. Plotting

In [ ]:
# 1) Segments number distribution
plt.figure(figsize=(3, 4))
ax = plt.gca()
sns.barplot(x=["18-core"] * len(df_segments_18), y=df_segments_18["N_Segments"], color='skyblue', capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, x=["18-core"] * len(df_segments_18), y=df_segments_18["N_Segments"],
                   dodge=False, size=2)
ax.set_title("Distribution of segment numbers (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of segments", fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Add label over error bar
vals = df_segments_18["N_Segments"]
m, s = vals.mean(), vals.sem()
yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
ax.text(0, max(m + s, vals.max()) + 0.01 * yrange, f"{m:.0f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("out/epi_18_core_segments_dist.png", bbox_inches="tight")
plt.close()


# Custom sort for 18-state names (e.g., 1_TssA, 10_TssB, 2_TssAFlnk)
def sort_states_18(states):
    return sorted(states, key=lambda x: int(x.split('_')[0]) if '_' in x and x.split('_')[0].isdigit() else 999)


# Sort states and prepare colors
all_states_18 = sort_states_18(df_comp_18['State'].unique())

# Fallback to summary_plots.STATE_COLORS if some colors are missing or black
for s in all_states_18:
    if s not in state_colors_18 or state_colors_18[s] == "#000000":
        # Try prefix match in summary_plots.STATE_COLORS
        name_part = s.split('_')[1] if '_' in s else s
        found_color = None
        for canonical, rgb in summary_plots.STATE_COLORS.items():
            if name_part.startswith(canonical):
                found_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                break
        if found_color:
            state_colors_18[s] = found_color
        elif s not in state_colors_18:
            state_colors_18[s] = "#888888"

# Create color list for pandas stacked bar plot (maintains order)
colors_18 = [state_colors_18.get(s, "#888888") for s in all_states_18]

# 2) State composition per dataset
pivot_comp_18 = df_comp_18.pivot(index='Dataset', columns='State', values='Fraction').fillna(0)
pivot_comp_18 = pivot_comp_18[all_states_18]  # Reorder columns
plt.figure(figsize=(15, 6))
ax = pivot_comp_18.plot(kind='bar', stacked=True, ax=plt.gca(), width=0.8, color=colors_18, linewidth=0)
pivot_comp_18.sum(axis=1).plot(kind='bar', ax=ax, width=0.8, facecolor='none', edgecolor="lightgrey", linewidth=1, legend=False)
ax.set_title("State composition per dataset (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='x-small', title="State")
ax.set_xlabel("Dataset", fontsize=9)
ax.set_ylabel("Fraction of Genome", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_18_core_composition_per_dataset.png", bbox_inches="tight")
plt.close()

# 3) Average composition
# Ensure all states are present for each Dataset, filling missing Fractions with 0
df_comp_18_filled = df_comp_18.pivot_table(index='Dataset', columns='State', values='Fraction', fill_value=0).stack().reset_index(name='Fraction')
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(10, 6),
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.06},
)

for ax in (ax_top, ax_bot):
    sns.barplot(data=df_comp_18_filled, x="State", y="Fraction", hue="State", order=all_states_18,
                hue_order=all_states_18, palette=state_colors_18, legend=False,
                capsize=0.1, errorbar="se", edgecolor="lightgrey", linewidth=1, ax=ax)
    utils.strip_points(ax, data=df_comp_18_filled, x="State", y="Fraction", hue="State",
                       order=all_states_18, hue_order=all_states_18,
                       dodge=False, size=1.5, alpha=0.4, jitter=0.2)

ax_top.set_ylim(BREAK_HIGH, 1.02)
ax_bot.set_ylim(0, BREAK_LOW)

# Hide the inner spines
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", bottom=False)

# Draw diagonal break marks
d = 0.012
kwargs = dict(transform=fig.transFigure, color="k", clip_on=False, linewidth=0.8)
for ax, sign in [(ax_top, -1), (ax_bot, 1)]:
    x0, x1 = ax.get_position().x0, ax.get_position().x1
    y = ax.get_position().y0 if sign == 1 else ax.get_position().y1
    for x in (x0, x1):
        fig.add_artist(plt.Line2D([x - d, x + d], [y + sign * d * 1.5, y - sign * d * 1.5], **kwargs))

for ax in (ax_top, ax_bot):
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis="y", labelsize=8)

ax_bot.set_xticklabels(all_states_18, rotation=45, ha="right", fontsize=8)
ax_bot.set_xlabel("State", fontsize=9)
ax_top.set_xlabel("")
ax_top.set_title("Average state composition (18-state core K27ac)", fontsize=11, fontweight="bold")
ax_top.set_ylabel("")
ax_bot.set_ylabel("Average Fraction of Genome", fontsize=9)

plt.savefig("out/epi_18_core_avg_composition.png", bbox_inches="tight")
plt.close()

# 3b) Average state mean length
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df_comp_18, x='State', y='MeanLength', hue='State', order=all_states_18, palette=state_colors_18,
                 legend=False, capsize=0.1, errorbar="se", edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_comp_18, x='State', y='MeanLength', order=all_states_18,
                   dodge=False, size=2)
ax.set_yscale("log")
ax.set_title("Average state mean length (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean length (bp, log scale)", fontsize=9)
ax.set_xlabel("State", fontsize=9)
ax.set_xticklabels(all_states_18, rotation=45, ha="right", fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_18_core_avg_mean_length.png", bbox_inches="tight")
plt.show()

# 3c) Average state median length
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df_comp_18, x='State', y='MedianLength', hue='State', order=all_states_18, palette=state_colors_18,
                 legend=False, capsize=0.1, errorbar="se", edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_comp_18, x='State', y='MedianLength', order=all_states_18,
                   dodge=False, size=2)
ax.set_yscale("log")
ax.set_title("Average state median length (18-state core K27ac)", fontsize=11, fontweight="bold")
ax.set_ylabel("Median length (bp, log scale)", fontsize=9)
ax.set_xlabel("State", fontsize=9)
ax.set_xticklabels(all_states_18, rotation=45, ha="right", fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_18_core_avg_median_length.png", bbox_inches="tight")
plt.show()

# 4) Pairwise consistency plots
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        fig, ax = plt.subplots(figsize=(3, 4))
        df_mode = df_pw_18[df_pw_18["Mode"] == mode]
        sns.barplot(x=["18-core"] * len(df_mode), y=df_mode[metric], color='skyblue', capsize=0.1, errorbar="se",
                    ax=ax, edgecolor="lightgrey", linewidth=1)
        utils.strip_points(ax, x=["18-core"] * len(df_mode), y=df_mode[metric],
                           dodge=False, size=1, alpha=0.25)
        ax.set_title(f"Pairwise {metric} consistency ({mode}) - 18-core", fontsize=11, fontweight="bold")
        ax.set_ylabel(f"{metric} index", fontsize=9)
        ax.grid(axis="y", alpha=0.3)

        # Add label over error bar
        m, s = df_mode[metric].mean(), df_mode[metric].sem()
        ax.text(0, utils.bar_label_y(ax, m + s, df_mode[metric].max()), f"{m:.2f}", ha="center", va="bottom", fontsize=8)

        fig.tight_layout()
        fig.savefig(f"out/epi_18_core_pw_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
        plt.close(fig)

# 5) Transition entropy plots
for mode in ["FULL", "NOQH"]:
    plt.figure(figsize=(3, 4))
    ax = plt.gca()
    df_mode = df_entropy_18[df_entropy_18["Mode"] == mode]
    sns.barplot(x=["18-core"] * len(df_mode), y=df_mode["Entropy"], color='skyblue', capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
    utils.strip_points(ax, x=["18-core"] * len(df_mode), y=df_mode["Entropy"],
                       dodge=False, size=2)
    mode_label = "(Full)" if mode == "FULL" else "(Excl. Quies/Het)"
    ax.set_title(f"Transition Entropy {mode_label}\n(18-state core K27ac)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Entropy (bits)", fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Add label over error bar
    m, s = df_mode["Entropy"].mean(), df_mode["Entropy"].sem()
    yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
    ax.text(0, max(m + s, df_mode["Entropy"].max()) + 0.01 * yrange, f"{m:.3f}", ha="center", va="bottom", fontsize=8)

    plt.tight_layout()
    plt.savefig(f"out/epi_18_core_entropy_{mode.lower()}.png", bbox_inches="tight")
    plt.close()


In [ ]:
# 6) State-by-state consistency heatmaps
print("Computing 18-state state-by-state consistency heatmaps...")
for mode in ["full", "noqh"]:
    for metric in ["Jaccard", "Kappa"]:
        excl = {"18_Quies", "13_Het"} if mode == "noqh" else set()
        avail_states = [s for s in all_states_18 if s not in excl]
        
        # Cells are averaged only over the pairs where they are *defined*. A
        # state absent from both segmentations of a pair carries no information;
        # averaging it in as 0.0 (the old behaviour) dragged the mean down.
        sum_mat = np.zeros((len(avail_states), len(avail_states)))
        n_mat = np.zeros((len(avail_states), len(avail_states)))
        count = 0
        
        for overlap in overlaps_18:
            # Restricted marginals and total area for the current set of states
            total_area = sum(overlap.get((s1, s2), 0) for s1 in avail_states for s2 in avail_states)
            if total_area == 0: continue
            a1_s = {s: sum(overlap.get((s, s2), 0) for s2 in avail_states) for s in avail_states}
            a2_s = {s: sum(overlap.get((s1, s), 0) for s1 in avail_states) for s in avail_states}
            
            mat = np.full((len(avail_states), len(avail_states)), np.nan)
            if metric == "Jaccard":
                for row_idx, s1 in enumerate(avail_states):
                    for col_idx, s2 in enumerate(avail_states):
                        ov = overlap.get((s1, s2), 0)
                        union = a1_s[s1] + a2_s[s2] - ov
                        if union > 0:
                            mat[row_idx, col_idx] = ov / union
            else: # Kappa
                for row_idx, s1 in enumerate(avail_states):
                    p1 = a1_s[s1] / total_area
                    for col_idx, s2 in enumerate(avail_states):
                        p2 = a2_s[s2] / total_area
                        p12 = overlap.get((s1, s2), 0) / total_area
                        denom = p1 + p2 - 2 * p1 * p2
                        if denom > 0:
                            mat[row_idx, col_idx] = 2 * (p12 - p1 * p2) / denom
            
            defined = ~np.isnan(mat)
            sum_mat[defined] += mat[defined]
            n_mat += defined
            count += 1
        
        if count > 0:
            avg_mat = np.where(n_mat > 0, sum_mat / np.maximum(n_mat, 1), np.nan)
            n_diag = np.diag(n_mat).astype(int)
            if n_diag.min() < count:
                thin = {s: int(n) for s, n in zip(avail_states, n_diag) if n < count}
                print(f"  18-state core {mode.upper()} {metric}: diagonal averaged over "
                      f"fewer than {count} pairs where the state was absent from both "
                      f"segmentations: {thin}")
            plt.figure(figsize=(10, 8))
            sns.heatmap(avg_mat, annot=True, fmt=".2f",
                        cmap="Reds" if metric == "Jaccard" else "RdBu_r",
                        center=0 if metric == "Kappa" else None,
                        xticklabels=avail_states, yticklabels=avail_states,
                        annot_kws={"size": 6})
            subtitle = ("one-vs-rest 2x2 kappa per state, own 0-1 scale"
                        if metric == "Kappa" else
                        "pairwise Jaccard per state pair")
            plt.title(f"18-state core Average {metric} Matrix ({mode.upper()})\n{subtitle}",
                      fontsize=10)
            plt.xlabel("State (Segmentation 2)")
            plt.ylabel("State (Segmentation 1)")
            plt.tight_layout()
            plt.savefig(f"out/pw_heatmap_18_core_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
            plt.close()


### 3. Show images

In [ ]:
for img in ["epi_18_core_segments_dist.png", "epi_18_core_composition_per_dataset.png", 
            "epi_18_core_avg_composition.png", "epi_18_core_avg_mean_length.png", 
            "epi_18_core_avg_median_length.png"]:
    path = f"out/{img}"
    if os.path.exists(path):
        display(Image(filename=path))
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        path = f"out/epi_18_core_pw_{mode.lower()}_{metric.lower()}.png"
        if os.path.exists(path):
            display(Image(filename=path))
for mode in ["full", "noqh"]:
    path = f"out/epi_18_core_entropy_{mode.lower()}.png"
    if os.path.exists(path):
        display(Image(filename=path))
for mode in ["full", "noqh"]:
    for metric in ["jaccard", "kappa"]:
        path = f"out/pw_heatmap_18_core_{mode}_{metric}.png"
        if os.path.exists(path):
            print(f"#### {metric.capitalize()} ({mode.upper()})")
            display(Image(filename=path))

# 15 states core models joint reordered vs inidividual matched

### 1. Identify files and compute/load stats

In [ ]:
# 1. Identify files
joint_15_files = sorted(glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_joint_reodered.bed.gz"))
indiv_15_files = sorted(glob.glob(f"{EPI_1000_PATH}/E*_15_coreMarks_dense_matched.bed"))

joint_15_ids_map = {os.path.basename(f).split("_")[0]: f for f in joint_15_files}
indiv_15_ids_map = {os.path.basename(f).split("_")[0]: f for f in indiv_15_files}

common_15_ids = sorted(list(set(joint_15_ids_map.keys()) & set(indiv_15_ids_map.keys())))
print(f"Found {len(common_15_ids)} common samples for 15-state models")

def get_file_stats_15_raw(f, segs):
    """Stats of one 15-state segmentation - see analyze.segmentation_stats."""
    stats = analyze.segmentation_stats(segs, 200, background=NOQH_15)
    return {"Dataset": os.path.basename(f).split("_")[0], **stats}


# 2. Computation / Loading
cache_15_path = "out/stats_15_cache.pkl"
# ! rm {cache_15_path}
if os.path.exists(cache_15_path):
    with open(cache_15_path, "rb") as f_cache:
        cache_15 = pickle.load(f_cache)
    # Validate cache has new columns
    if cache_15 and not all("entropy" in r for r in cache_15.values() if r.get("composition")):
        print("15-state cache outdated (missing entropy), recomputing...")
        cache_15 = {}
else:
    cache_15 = {}

results_15 = []
updated_15 = False

print(f"Computing stats for 15-state models...")
for eid in tqdm(common_15_ids):
    for t_name, f_map in [("Joint", joint_15_ids_map), ("Individual", indiv_15_ids_map)]:
        key = (eid, t_name)
        if key in cache_15:
            stats = cache_15[key]
        else:
            f_path = f_map[eid]
            segs = match.load_bed(f_path)
            stats = get_file_stats_15_raw(f_path, segs)
            stats["Type"] = t_name
            cache_15[key] = stats
            updated_15 = True
        results_15.append(stats)

if updated_15:
    os.makedirs(os.path.dirname(cache_15_path), exist_ok=True)
    with open(cache_15_path, "wb") as f_cache:
        pickle.dump(cache_15, f_cache)

df_segments_15 = pd.DataFrame([{"Dataset": r["Dataset"], "Type": r["Type"], "N_Segments": r["n_segments"]} for r in results_15])
df_comp_15 = pd.DataFrame([{"Dataset": r["Dataset"], "Type": r["Type"], "State": c["State"], 
                            "Fraction": c["Fraction"], "MeanLength": c["MeanLength"], "MedianLength": c["MedianLength"]} 
                            for r in results_15 for c in r["composition"]])
df_entropy_15 = pd.DataFrame([{"Dataset": r["Dataset"], "Type": r["Type"], "Mode": m.upper(), "Entropy": val}
                               for r in results_15 for m, val in r["entropy"].items()])
state_colors_15 = {}
for r in results_15:
    state_colors_15.update(r["colors"])

# Save aggregated results
df_segments_15.to_csv("out/df_segments_15.csv", index=False)
df_comp_15.to_csv("out/df_comp_15.csv", index=False)
df_entropy_15.to_csv("out/df_entropy_15.csv", index=False)
with open("out/state_colors_15.json", "w") as f:
    json.dump(state_colors_15, f)


### 4. Comparison of Joint vs Individual models for the same sample

In [ ]:
def compute_joint_indiv_metrics(s1_path, s2_path):
    """Agreement of an individually vs jointly called segmentation of one sample."""
    s1, s2 = match.load_bed(s1_path), match.load_bed(s2_path)
    l1, l2 = match.state_lengths(s1), match.state_lengths(s2)
    # These segmentations mix naming conventions ("Quies" and "15_Quies"), so the
    # background is matched by substring rather than by exact state name.
    background = {s for s in (set(l1) | set(l2))
                  if any(bg in s for bg in NOQH_DENOVO)}
    return match.agreement_by_mode(match.pair_overlap(s1, s2), l1, l2,
                                   background=background)


caller_tasks = {}
for folder, method_name, path, bin_size in valid_tasks:
    caller_tasks[(folder, method_name)] = path

comp_tasks = []
# Reference 15-state (ChromHMM)
for eid in common_15_ids:
    comp_tasks.append(("ChromHMM", eid, indiv_15_ids_map[eid], joint_15_ids_map[eid]))

# Peak callers
for caller in ["HOMER", "MACS2", "Omnipeak"]:
    c_indiv = caller
    c_joint = f"Joint {caller}"
    common_folders = set([f for f, m in caller_tasks.keys() if m == c_indiv]) & \
                     set([f for f, m in caller_tasks.keys() if m == c_joint])
    for folder in sorted(list(common_folders)):
        comp_tasks.append((caller, folder, caller_tasks[(folder, c_indiv)], caller_tasks[(folder, c_joint)]))

def compute_joint_indiv():
    print(f"Computing Joint vs Individual metrics for {len(comp_tasks)} pairs...")
    joint_indiv_results = []
    for method, eid, path_indiv, path_joint in tqdm(comp_tasks):
        metrics = compute_joint_indiv_metrics(path_indiv, path_joint)
        for mode in ["full", "noqh"]:
            row = {"Method": method, "Dataset": eid, "Mode": mode.upper()}
            row.update(metrics[mode])
            joint_indiv_results.append(row)
    return pd.DataFrame(joint_indiv_results)

df_joint_indiv = utils.cached_csv(
    "out/df_joint_indiv_comparison.csv", compute_joint_indiv, label="Joint vs Individual metrics",
    valid=lambda df: len(df.groupby(["Method", "Dataset"])) == len(comp_tasks))

# Visualization
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa", "Cosine"]:
        df_mode = df_joint_indiv[df_joint_indiv["Mode"] == mode]
        if df_mode.empty: continue
        
        fig, ax = plt.subplots(figsize=(6, 4.2))
        sns.barplot(data=df_mode, x="Method", y=metric, hue="Method", palette=method_palette,
                    order=["ChromHMM", "HOMER", "MACS2", "Omnipeak"], 
                    capsize=0.05, errorbar="se", err_kws={"linewidth": 2.0}, dodge=False, ax=ax, edgecolor="lightgrey", linewidth=1)
        utils.strip_points(ax, data=df_mode, x="Method", y=metric,
                           order=["ChromHMM", "HOMER", "MACS2", "Omnipeak"],
                           dodge=False, size=2)
        
        ax.set_title(f"Joint vs Individual ({mode}): {metric}", fontsize=11, fontweight="bold")
        ax.set_ylabel(metric, fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
        
        # Add labels over error bars
        method_order_plot = ["ChromHMM", "HOMER", "MACS2", "Omnipeak"]
        yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
        for i, method in enumerate(method_order_plot):
            subset = df_mode[df_mode["Method"] == method][metric]
            if subset.empty: continue
            m, s = subset.mean(), subset.sem()
            if pd.isna(m): continue
            top = max(m + (s if not pd.isna(s) else 0), subset.max())
            ax.text(i, top + yrange * 0.01, f"{m:.2f}", ha="center", va="bottom", fontsize=6)
            
        fig.tight_layout()
        plot_path = f"out/joint_indiv_comparison_{mode.lower()}_{metric.lower()}.png"
        fig.savefig(plot_path, bbox_inches="tight")
        display(Image(filename=plot_path))
        plt.close(fig)


In [ ]:
# 3. Pairwise metrics computation
df_15_states_path = "out/df_pw_15.csv"
cache_pw_15_path = "out/pw_15_cache.pkl"

if os.path.exists(cache_pw_15_path):
    with open(cache_pw_15_path, "rb") as f_cache:
        cache_pw_15 = pickle.load(f_cache)
else:
    cache_pw_15 = {}

pw_data_15 = []
overlaps_15 = {}
updated_pw_15 = False

for t_name, f_map in [("Individual", indiv_15_ids_map), ("Joint", joint_15_ids_map)]:
    print(f"Checking pairwise overlaps for 15-state {t_name} (1000 pairs)...")
    current_ids = common_15_ids
    pairs = [tuple(sorted((id1, id2))) for i, id1 in enumerate(current_ids) for id2 in current_ids[i+1:]]
    if len(pairs) > 1000:
        random.seed(42)
        pairs = random.sample(pairs, 1000)
        pairs.sort()

    overlaps_15[t_name] = []
    last_id1 = None
    s1, l1 = None, None
    for id1, id2 in tqdm(pairs, desc=f"Overlaps {t_name}"):
        key = (t_name, id1, id2)
        overlap = None
        if key in cache_pw_15:
            cached_val = cache_pw_15[key]
            if isinstance(cached_val, dict) and 'overlap' in cached_val:
                metrics = cached_val['metrics']
                overlap = cached_val['overlap']
            else:
                metrics = cached_val
        else:
            if id1 != last_id1:
                s1 = match.load_bed(f_map[id1])
                l1 = match.state_lengths(s1)
                last_id1 = id1
            s2 = match.load_bed(f_map[id2])
            l2 = match.state_lengths(s2)
            overlap = match.pair_overlap(s1, s2)
            # Tuples, matching what out/pw_15_cache.pkl already holds.
            metrics = {mode: (m["Jaccard"], m["Kappa"]) for mode, m in
                       match.agreement_by_mode(overlap, l1, l2, background=NOQH_15).items()}
            cache_pw_15[key] = {'metrics': metrics, 'overlap': overlap}
            updated_pw_15 = True

        if overlap is None:
            # Recompute overlap if it wasn't in cache
            if id1 != last_id1:
                s1 = match.load_bed(f_map[id1])
                l1 = match.state_lengths(s1)
                last_id1 = id1
            s2 = match.load_bed(f_map[id2])
            overlap = match.pair_overlap(s1, s2)
            # Update cache to include overlap
            cache_pw_15[key] = {'metrics': metrics, 'overlap': overlap}
            updated_pw_15 = True

        overlaps_15[t_name].append(overlap)

        for mode, (jaccard, kappa) in metrics.items():
            pw_data_15.append({"Type": t_name, "Dataset1": id1, "Dataset2": id2,
                               "Mode": mode.upper(), "Jaccard": jaccard, "Kappa": kappa})

if updated_pw_15:
    os.makedirs(os.path.dirname(cache_pw_15_path), exist_ok=True)
    with open(cache_pw_15_path, "wb") as f_cache:
        pickle.dump(cache_pw_15, f_cache)

df_pw_15 = pd.DataFrame(pw_data_15)
df_pw_15.to_csv(df_15_states_path, index=False)


### 2. Plotting 15-state models

In [ ]:
# 1) Segments number distribution
plt.figure(figsize=(4, 4))
ax = plt.gca()
sns.barplot(data=df_segments_15, x="Type", y="N_Segments", order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"}, capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_segments_15, x="Type", y="N_Segments",
                   order=["Individual", "Joint"], dodge=False, size=2)
ax.set_title("Distribution of segment numbers (15-state)", fontsize=11, fontweight="bold")
ax.set_ylabel("Number of segments", fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Add labels over error bars
for i, t in enumerate(["Individual", "Joint"]):
    subset = df_segments_15[df_segments_15["Type"] == t]
    if not subset.empty:
        m, s = subset["N_Segments"].mean(), subset["N_Segments"].sem()
        yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
        ax.text(i, max(m + s, subset["N_Segments"].max()) + 0.01 * yrange, f"{m:.0f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("out/epi_15_segments_comparison.png", bbox_inches="tight")
plt.close()

In [ ]:
# Custom sort for 15-state names
def sort_states_15(states):
    return sorted(states, key=lambda x: int(x.split('_')[0]) if '_' in x and x.split('_')[0].isdigit() else 999)

all_states_15 = sort_states_15(df_comp_15['State'].unique())

# 2) Average state composition for both types on a single plot
# Fallback to summary_plots.STATE_COLORS if some colors are missing or black
for s in all_states_15:
    if s not in state_colors_15 or state_colors_15[s] == "#000000":
        # Try prefix match in summary_plots.STATE_COLORS
        name_part = s.split('_')[1] if '_' in s else s
        found_color = None
        for canonical, rgb in summary_plots.STATE_COLORS.items():
            if name_part.startswith(canonical):
                found_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
                break
        if found_color:
            state_colors_15[s] = found_color
        elif s not in state_colors_15:
            state_colors_15[s] = "#888888"

colors_15_list = [state_colors_15.get(s, "#888888") for s in all_states_15]

# Ensure all states are present for each (Type, Dataset) pair, filling missing Fractions with 0
df_comp_15_filled = df_comp_15.pivot_table(index=['Type', 'Dataset'], columns='State', values='Fraction', fill_value=0).stack().reset_index(name='Fraction')

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, sharex=True,
    figsize=(12, 6),
    gridspec_kw={"height_ratios": [1, 4], "hspace": 0.06},
)

for ax in (ax_top, ax_bot):
    sns.barplot(data=df_comp_15_filled, x="State", y="Fraction", hue="Type", order=all_states_15,
                hue_order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"},
                capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1,
                legend=(ax is ax_top))
    utils.strip_points(ax, data=df_comp_15_filled, x="State", y="Fraction", hue="Type",
                       order=all_states_15, hue_order=["Individual", "Joint"],
                       size=1.5, alpha=0.4, jitter=0.2)

ax_top.set_ylim(BREAK_HIGH, 1.02)
ax_bot.set_ylim(0, BREAK_LOW)

# Hide the inner spines
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", bottom=False)

# Draw diagonal break marks
d = 0.012
kwargs = dict(transform=fig.transFigure, color="k", clip_on=False, linewidth=0.8)
for ax, sign in [(ax_top, -1), (ax_bot, 1)]:
    x0, x1 = ax.get_position().x0, ax.get_position().x1
    y = ax.get_position().y0 if sign == 1 else ax.get_position().y1
    for x in (x0, x1):
        fig.add_artist(plt.Line2D([x - d, x + d], [y + sign * d * 1.5, y - sign * d * 1.5], **kwargs))

for ax in (ax_top, ax_bot):
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis="y", labelsize=8)

ax_bot.set_xticklabels(all_states_15, rotation=45, ha="right", fontsize=8)
ax_bot.set_xlabel("State", fontsize=9)
ax_top.set_xlabel("")
ax_top.set_title("Average state composition comparison (15-state)", fontsize=11, fontweight="bold")
ax_top.set_ylabel("")
ax_bot.set_ylabel("Average Fraction of Genome", fontsize=9)

if ax_top.get_legend():
    ax_top.legend(title="Type", fontsize=8, title_fontsize=9,
                  bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

plt.savefig("out/epi_15_avg_composition_comparison.png", bbox_inches="tight")
plt.close()

# 2d) Average state composition comparison (stacked)
pivot_avg_15 = df_comp_15.pivot_table(index=['Type', 'Dataset'], columns='State', values='Fraction', fill_value=0).groupby('Type').mean()
pivot_avg_15 = pivot_avg_15.reindex(index=["Individual", "Joint"], columns=all_states_15)

plt.figure(figsize=(8, 6))
ax = pivot_avg_15.plot(kind='bar', stacked=True, ax=plt.gca(), width=0.6, color=colors_15_list, linewidth=0)
pivot_avg_15.sum(axis=1).plot(kind='bar', ax=ax, width=0.6, facecolor='none', edgecolor="lightgrey", linewidth=1, legend=False)
ax.set_title("Average state composition comparison (15-state, stacked)", fontsize=11, fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='small', title="State")
ax.set_xlabel("Method", fontsize=9)
ax.set_ylabel("Average Fraction of Genome", fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_15_avg_composition_stacked.png", bbox_inches="tight")
plt.close()

# 2b) Average state mean length comparison
plt.figure(figsize=(12, 6))
ax = plt.gca()
sns.barplot(data=df_comp_15, x="State", y="MeanLength", hue="Type", order=all_states_15,
            hue_order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"},
            capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_comp_15, x="State", y="MeanLength", hue="Type",
                   order=all_states_15, hue_order=["Individual", "Joint"],
                   size=1.5, alpha=0.4, jitter=0.2)
ax.set_yscale("log")
ax.set_title("Average state mean length comparison (15-state)", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean length (bp, log scale)", fontsize=9)
ax.set_xlabel("State", fontsize=9)
ax.set_xticklabels(all_states_15, rotation=45, ha="right", fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_15_avg_mean_length_comparison.png", bbox_inches="tight")
plt.show()

# 2c) Average state median length comparison
plt.figure(figsize=(12, 6))
ax = plt.gca()
sns.barplot(data=df_comp_15, x="State", y="MedianLength", hue="Type", order=all_states_15,
            hue_order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"},
            capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
utils.strip_points(ax, data=df_comp_15, x="State", y="MedianLength", hue="Type",
                   order=all_states_15, hue_order=["Individual", "Joint"],
                   size=1.5, alpha=0.4, jitter=0.2)
ax.set_yscale("log")
ax.set_title("Average state median length comparison (15-state)", fontsize=11, fontweight="bold")
ax.set_ylabel("Median length (bp, log scale)", fontsize=9)
ax.set_xlabel("State", fontsize=9)
ax.set_xticklabels(all_states_15, rotation=45, ha="right", fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("out/epi_15_avg_median_length_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# 3) For each type state composition for each sample
for t in ["Individual", "Joint"]:
    pivot_comp = df_comp_15[df_comp_15["Type"] == t].pivot(index='Dataset', columns='State', values='Fraction').fillna(0)
    pivot_comp = pivot_comp.reindex(columns=all_states_15)
    plt.figure(figsize=(15, 6))
    ax = pivot_comp.plot(kind='bar', stacked=True, ax=plt.gca(), width=0.8, color=colors_15_list, linewidth=0)
    pivot_comp.sum(axis=1).plot(kind='bar', ax=ax, width=0.8, facecolor='none', edgecolor="lightgrey", linewidth=1, legend=False)
    ax.set_title(f"State composition per dataset ({t} 15-state)", fontsize=11, fontweight="bold")
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize='x-small', title="State")
    ax.set_xlabel("Dataset", fontsize=9)
    ax.set_ylabel("Fraction of Genome", fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=6)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"out/epi_15_{t.lower()}_composition_per_dataset.png", bbox_inches="tight")
    plt.close()

In [ ]:
# 4) Pairwise consistency plots
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        plt.figure(figsize=(4, 4))
        ax = plt.gca()
        sns.barplot(data=df_pw_15[df_pw_15["Mode"] == mode], x="Type", y=metric, 
                    order=["Individual", "Joint"], palette={"Joint": "skyblue", "Individual": "lightcoral"},
                    capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
        utils.strip_points(ax, data=df_pw_15[df_pw_15["Mode"] == mode], x="Type", y=metric,
                           order=["Individual", "Joint"], dodge=False,
                           size=1, alpha=0.25)
        ax.set_title(f"Pairwise {metric} consistency ({mode})\n(15-state models)", fontsize=11, fontweight="bold")
        ax.set_ylabel(f"{metric} index", fontsize=9)
        ax.grid(axis='y', alpha=0.3)
        
        # Add labels over error bars
        for i, t in enumerate(["Individual", "Joint"]):
            subset = df_pw_15[(df_pw_15["Type"] == t) & (df_pw_15["Mode"] == mode)]
            if not subset.empty:
                m, s = subset[metric].mean(), subset[metric].sem()
                ax.text(i, utils.bar_label_y(ax, m + s, subset[metric].max()), f"{m:.2f}", ha="center", va="bottom", fontsize=8)
        
        plt.tight_layout()
        plt.savefig(f"out/epi_15_pw_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
        plt.close()

# 5) Transition matrix entropy comparison
for mode in ["FULL", "NOQH"]:
    plt.figure(figsize=(4, 4))
    ax = plt.gca()
    df_mode = df_entropy_15[df_entropy_15["Mode"] == mode]
    sns.barplot(data=df_mode, x="Type", y="Entropy", order=["Individual", "Joint"], 
                palette={"Joint": "skyblue", "Individual": "lightcoral"}, capsize=0.1, errorbar="se", ax=ax, edgecolor="lightgrey", linewidth=1)
    utils.strip_points(ax, data=df_mode, x="Type", y="Entropy",
                       order=["Individual", "Joint"], dodge=False, size=2)
    mode_label = "(Full)" if mode == "FULL" else "(Excl. Quies/Het)"
    ax.set_title(f"Transition matrix entropy {mode_label}\n(15-state models)", fontsize=11, fontweight="bold")
    ax.set_ylabel("Entropy (bits)", fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    
    # Add labels over error bars
    for i, t in enumerate(["Individual", "Joint"]):
        subset = df_mode[df_mode["Type"] == t]
        if not subset.empty:
            m, s = subset["Entropy"].mean(), subset["Entropy"].sem()
            yrange = ax.get_ylim()[1] - ax.get_ylim()[0]
            ax.text(i, max(m + s, subset["Entropy"].max()) + 0.01 * yrange, f"{m:.3f}", ha="center", va="bottom", fontsize=8)
            
    plt.tight_layout()
    plt.savefig(f"out/epi_15_entropy_{mode.lower()}.png", bbox_inches="tight")
    plt.close()


In [ ]:
# 6) State-by-state consistency heatmaps for 15-state
print("Computing 15-state state-by-state consistency heatmaps...")
for t_name in ["Individual", "Joint"]:
    for mode in ["full", "noqh"]:
        for metric in ["Jaccard", "Kappa"]:
            excl = {"15_Quies", "9_Het"} if mode == "noqh" else set()
            avail_states = [s for s in all_states_15 if s not in excl]
            
            # Cells are averaged only over the pairs where they are *defined*. A
            # state absent from both segmentations of a pair carries no information;
            # averaging it in as 0.0 (the old behaviour) dragged the mean down.
            sum_mat = np.zeros((len(avail_states), len(avail_states)))
            n_mat = np.zeros((len(avail_states), len(avail_states)))
            count = 0
            
            for overlap in overlaps_15[t_name]:
                # Restricted marginals and total area for the current set of states
                total_area = sum(overlap.get((s1, s2), 0) for s1 in avail_states for s2 in avail_states)
                if total_area == 0: continue
                a1_s = {s: sum(overlap.get((s, s2), 0) for s2 in avail_states) for s in avail_states}
                a2_s = {s: sum(overlap.get((s1, s), 0) for s1 in avail_states) for s in avail_states}
                
                mat = np.full((len(avail_states), len(avail_states)), np.nan)
                if metric == "Jaccard":
                    for row_idx, s1 in enumerate(avail_states):
                        for col_idx, s2 in enumerate(avail_states):
                            ov = overlap.get((s1, s2), 0)
                            union = a1_s[s1] + a2_s[s2] - ov
                            if union > 0:
                                mat[row_idx, col_idx] = ov / union
                else: # Kappa
                    for row_idx, s1 in enumerate(avail_states):
                        p1 = a1_s[s1] / total_area
                        for col_idx, s2 in enumerate(avail_states):
                            p2 = a2_s[s2] / total_area
                            p12 = overlap.get((s1, s2), 0) / total_area
                            denom = p1 + p2 - 2 * p1 * p2
                            if denom > 0:
                                mat[row_idx, col_idx] = 2 * (p12 - p1 * p2) / denom
                
                defined = ~np.isnan(mat)
                sum_mat[defined] += mat[defined]
                n_mat += defined
                count += 1
            
            if count > 0:
                avg_mat = np.where(n_mat > 0, sum_mat / np.maximum(n_mat, 1), np.nan)
                n_diag = np.diag(n_mat).astype(int)
                if n_diag.min() < count:
                    thin = {s: int(n) for s, n in zip(avail_states, n_diag) if n < count}
                    print(f"  15-state {t_name} {mode.upper()} {metric}: diagonal averaged over "
                          f"fewer than {count} pairs where the state was absent from both "
                          f"segmentations: {thin}")
                plt.figure(figsize=(10, 8))
                sns.heatmap(avg_mat, annot=True, fmt=".2f",
                            cmap="Reds" if metric == "Jaccard" else "RdBu_r",
                            center=0 if metric == "Kappa" else None,
                            xticklabels=avail_states, yticklabels=avail_states,
                            annot_kws={"size": 6})
                subtitle = ("one-vs-rest 2x2 kappa per state, own 0-1 scale"
                            if metric == "Kappa" else
                            "pairwise Jaccard per state pair")
                plt.title(f"15-state {t_name} Average {metric} Matrix ({mode.upper()})\n{subtitle}",
                          fontsize=10)
                plt.xlabel("State (Segmentation 2)")
                plt.ylabel("State (Segmentation 1)")
                plt.tight_layout()
                plt.savefig(f"out/pw_heatmap_15_{t_name.lower()}_{mode.lower()}_{metric.lower()}.png", bbox_inches="tight")
                plt.close()


### 3. Show 15-state analysis images

In [ ]:
for img in ["epi_15_segments_comparison.png", "epi_15_avg_composition_comparison.png", 
            "epi_15_avg_composition_stacked.png", "epi_15_avg_mean_length_comparison.png", 
            "epi_15_avg_median_length_comparison.png", "epi_15_individual_composition_per_dataset.png", 
            "epi_15_joint_composition_per_dataset.png"]:
    path = f"out/{img}"
    if os.path.exists(path):
        display(Image(filename=path))
for mode in ["FULL", "NOQH"]:
    for metric in ["Jaccard", "Kappa"]:
        path = f"out/epi_15_pw_{mode.lower()}_{metric.lower()}.png"
        if os.path.exists(path):
            display(Image(filename=path))
    path = f"out/epi_15_entropy_{mode.lower()}.png"
    if os.path.exists(path):
        display(Image(filename=path))
for t_name in ["Individual", "Joint"]:
    for mode in ["full", "noqh"]:
        for metric in ["jaccard", "kappa"]:
            path = f"out/pw_heatmap_15_{t_name.lower()}_{mode}_{metric}.png"
            if os.path.exists(path):
                print(f"#### {t_name} {metric.capitalize()} ({mode.upper()})")
                display(Image(filename=path))


# Cell type differences in chromatin

### 1. Compute

In [ ]:
os.makedirs("out/consistency", exist_ok=True)

# 1. De-novo methods
method_paths = defaultdict(list)
for task, path in zip(valid_tasks, all_segs):
    method_name = task[1]
    method_paths[method_name].append(path)

windows = [1000, 0]
denovo_counts = {w: {} for w in windows}
for method_name in ["ChromHMM", "HOMER", "MACS2", "Omnipeak", "Joint HOMER", "Joint MACS2", "Joint Omnipeak"]:
    if method_name not in method_paths: continue

    for w in windows:
        def compute_denovo_consistency(method_name=method_name, w=w):
            segs_list = [match.load_bed(p) for p in tqdm(method_paths[method_name],
                                                         desc="Loading BEDs", leave=False)]
            return analyze.compute_state_consistency(segs_list, window=w, show_progress=True)

        slug = f"denovo_{method_name.lower().replace(' ', '_')}"
        denovo_counts[w][method_name] = utils.cached_pickle(
            consistency_cache_path(slug, w), compute_denovo_consistency,
            label=f"consistency for de-novo {method_name} (w={w})")


In [ ]:
# 2. 15-state models
state_15_counts = {w: {} for w in windows}
for t_name, f_map in [("Individual", indiv_15_ids_map), ("Joint", joint_15_ids_map)]:
    for w in windows:
        def compute_15state_consistency(t_name=t_name, f_map=f_map, w=w):
            files = [f_map[eid] for eid in common_15_ids]
            segs_list = [match.load_bed(f) for f in tqdm(files, desc=f"Loading {t_name}")]
            return analyze.compute_state_consistency(segs_list, window=w, show_progress=True)

        state_15_counts[w][t_name] = utils.cached_pickle(
            consistency_cache_path(f"15state_{t_name.lower()}", w),
            compute_15state_consistency,
            label=f"consistency for 15-state {t_name} (w={w})")


In [ ]:
# 3. 18-state models
counts_18 = {}
for w in windows:
    def compute_18state_consistency(w=w):
        segs_18_list = [loaded_segs_18[eid] for eid in epi_18_core_ids]
        return analyze.compute_state_consistency(segs_18_list, window=w, show_progress=True)

    counts_18[w] = utils.cached_pickle(
        consistency_cache_path("18state_core", w), compute_18state_consistency,
        label=f"consistency for 18-state Core (w={w})")


### 2. Plotting

In [ ]:
# 1. De-novo methods
# The compute cells above cache every count under out/consistency/, so recover from
# disk when they have not been run in this kernel — running this cell on its own used
# to raise "NameError: denovo_counts". Anything already in memory is left untouched.
import json

DENOVO_METHODS = ["ChromHMM", "HOMER", "MACS2", "Omnipeak",
                  "Joint HOMER", "Joint MACS2", "Joint Omnipeak"]
if "windows" not in globals():
    windows = [1000, 0]


def load_consistency(slug):
    """{w: counts} from the consistency caches of *slug*; absent windows skipped."""
    loaded = {}
    for w in windows:
        path = consistency_cache_path(slug, w)
        if os.path.exists(path):
            with open(path, "rb") as f:
                loaded[w] = pickle.load(f)
        else:
            print(f"  WARNING: missing {path}")
    return loaded


def load_state_colors(path):
    if not os.path.exists(path):
        print(f"  WARNING: missing {path}, falling back to default colors")
        return {}
    with open(path) as f:
        return json.load(f)


def n_segmentations(counts):
    """Number of segmentations behind a counts dict — its deepest support bucket.

    compute_state_consistency() keys each state on depth 1..M, so this recovers M
    without needing the original file lists in memory (plot_state_consistency does
    the same thing internally).
    """
    return max(d for depths in counts.values() for d in depths)


if "denovo_counts" not in globals():
    print("denovo_counts not in memory - loading out/consistency/ caches")
    denovo_counts = {w: {} for w in windows}
    for m_name in DENOVO_METHODS:
        for w, counts in load_consistency(
                f"denovo_{m_name.lower().replace(' ', '_')}").items():
            denovo_counts[w][m_name] = counts

if "state_15_counts" not in globals():
    print("state_15_counts not in memory - loading out/consistency/ caches")
    state_15_counts = {w: {} for w in windows}
    for t_name in ["Individual", "Joint"]:
        for w, counts in load_consistency(f"15state_{t_name.lower()}").items():
            state_15_counts[w][t_name] = counts

if "counts_18" not in globals():
    print("counts_18 not in memory - loading out/consistency/ caches")
    counts_18 = load_consistency("18state_core")

if "method_paths" not in globals():
    if "valid_tasks" not in globals():
        raise RuntimeError("Run the setup cells at the top of the notebook first — "
                           "they define valid_tasks / all_segs.")
    method_paths = defaultdict(list)
    for task, path in zip(valid_tasks, all_segs):
        method_paths[task[1]].append(path)

if "state_colors_15" not in globals():
    state_colors_15 = load_state_colors("out/state_colors_15.json")
if "state_colors_18" not in globals():
    state_colors_18 = load_state_colors("out/state_colors_18.json")

for method_name in DENOVO_METHODS:
    if method_name not in denovo_counts.get(0, {}):
        continue
    print(f"--- Plotting de-novo {method_name} ---")
    M = n_segmentations(denovo_counts[0][method_name])
    for state in sorted(denovo_counts[0][method_name].keys(), key=analyze._natural_sort_key):
        print(f"  {state}: {denovo_counts[0][method_name][state][M]}")
    # Use actual colors from segmentations
    colors = match.state_colors(match.load_bed(method_paths[method_name][0]))

    for w in windows:
        if method_name not in denovo_counts.get(w, {}):
            continue
        analyze.plot_state_consistency(denovo_counts[w][method_name],
                                       f"De-novo {method_name} (w={w})",
                                       f"out/epi_denovo_{method_name.lower().replace(' ', '_')}_w{w}_consistency.png",
                                       colors=colors)

# 2. 15-state models
for t_name in ["Individual", "Joint"]:
    if t_name not in state_15_counts.get(0, {}):
        continue
    print(f"--- Plotting 15-state {t_name} ---")
    M = n_segmentations(state_15_counts[0][t_name])
    for state in sorted(state_15_counts[0][t_name].keys(), key=analyze._natural_sort_key):
        print(f"  {state}: {state_15_counts[0][t_name][state][M]}")

    for w in windows:
        if t_name not in state_15_counts.get(w, {}):
            continue
        analyze.plot_state_consistency(state_15_counts[w][t_name],
                                       f"15-state {t_name} (w={w})",
                                       f"out/epi_15_{t_name.lower()}_w{w}_consistency.png",
                                       colors=state_colors_15)

# 3. 18-state models
if counts_18.get(0):
    print("--- Plotting 18-state Core ---")
    M_18 = n_segmentations(counts_18[0])
    for state in sorted(counts_18[0].keys(), key=analyze._natural_sort_key):
        print(f"  {state}: {counts_18[0][state][M_18]}")

    for w in windows:
        if w not in counts_18:
            continue
        analyze.plot_state_consistency(counts_18[w],
                                       f"18-state Core (w={w})",
                                       f"out/epi_18_core_w{w}_consistency.png",
                                       colors=state_colors_18)


### 3. Display

In [ ]:
if "windows" not in globals():
    windows = [1000, 0]

for method_name in ["ChromHMM", "HOMER", "MACS2", "Omnipeak", "Joint HOMER", "Joint MACS2", "Joint Omnipeak"]:
    for w in windows:
        path = f"out/epi_denovo_{method_name.lower().replace(' ', '_')}_w{w}_consistency.png"
        if os.path.exists(path):
            display(Image(filename=path))
for t_name in ["Individual", "Joint"]:
    for w in windows:
        path = f"out/epi_15_{t_name.lower()}_w{w}_consistency.png"
        if os.path.exists(path):
            display(Image(filename=path))
for w in windows:
    path = f"out/epi_18_core_w{w}_consistency.png"
    if os.path.exists(path):
        display(Image(filename=path))